In [1]:
! pip install -q transformers 
! pip install -q sentence-transformers
! pip install -q polars
! pip install -q pyarrow
! pip install -q xgboost
! pip install -q lightgbm
! pip install -q catboost

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer

from transformers import DistilBertModel, DistilBertTokenizer
from transformers import LongformerTokenizer,LongformerModel
from sentence_transformers import SentenceTransformer

from typing import List

import pandas as pd
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path
import pyarrow as pa, gc
import sys
import time
import random

import os
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import warnings
import math
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit

import xgboost as xgb
from lightgbm import LGBMRanker, early_stopping, log_evaluation
from catboost import CatBoostRanker, Pool


pd.set_option('display.max_columns', None) 
pd.set_option('display.max_rows', None)

warnings.filterwarnings("ignore", category=RuntimeWarning)

In [3]:
trainPath = Path("/kaggle/input/aeroclub-recsys-2025/train.parquet")
testPath = Path("/kaggle/input/aeroclub-recsys-2025/test.parquet")

# Now read 1st parquet file 

In [4]:
# dont need : class PLDFPorcessorForNN() 
trainDFPl_base = pl.read_parquet(f"./dataset_part1.parquet")

trainDFPl_base.head()

Id,bySelf,companyID,frequentFlyer,nationality,isVip,legs0_arrivalAt,legs0_departureAt,legs0_duration,legs0_segments0_aircraft_code,legs0_segments0_arrivalTo_airport_city_iata,legs0_segments0_arrivalTo_airport_iata,legs0_segments0_baggageAllowance_quantity,legs0_segments0_baggageAllowance_weightMeasurementType,legs0_segments0_cabinClass,legs0_segments0_departureFrom_airport_iata,legs0_segments0_duration,legs0_segments0_marketingCarrier_code,legs0_segments0_operatingCarrier_code,legs0_segments0_seatsAvailable,legs0_segments1_aircraft_code,legs0_segments1_arrivalTo_airport_city_iata,legs0_segments1_arrivalTo_airport_iata,legs0_segments1_baggageAllowance_quantity,legs0_segments1_baggageAllowance_weightMeasurementType,legs0_segments1_cabinClass,legs0_segments1_departureFrom_airport_iata,legs0_segments1_duration,legs0_segments1_marketingCarrier_code,legs0_segments1_operatingCarrier_code,legs0_segments1_seatsAvailable,legs0_segments2_aircraft_code,legs0_segments2_arrivalTo_airport_city_iata,legs0_segments2_arrivalTo_airport_iata,legs0_segments2_baggageAllowance_quantity,legs0_segments2_baggageAllowance_weightMeasurementType,legs0_segments2_cabinClass,…,legs1_segments1_arrivalTo_airport_city_iata,legs1_segments1_arrivalTo_airport_iata,legs1_segments1_baggageAllowance_quantity,legs1_segments1_baggageAllowance_weightMeasurementType,legs1_segments1_cabinClass,legs1_segments1_departureFrom_airport_iata,legs1_segments1_duration,legs1_segments1_marketingCarrier_code,legs1_segments1_operatingCarrier_code,legs1_segments1_seatsAvailable,legs1_segments2_aircraft_code,legs1_segments2_arrivalTo_airport_city_iata,legs1_segments2_arrivalTo_airport_iata,legs1_segments2_baggageAllowance_quantity,legs1_segments2_baggageAllowance_weightMeasurementType,legs1_segments2_cabinClass,legs1_segments2_departureFrom_airport_iata,legs1_segments2_duration,legs1_segments2_marketingCarrier_code,legs1_segments2_operatingCarrier_code,legs1_segments2_seatsAvailable,legs1_segments3_aircraft_code,legs1_segments3_marketingCarrier_code,legs1_segments3_operatingCarrier_code,miniRules0_monetaryAmount,miniRules0_statusInfos,miniRules1_monetaryAmount,miniRules1_statusInfos,pricingInfo_isAccessTP,pricingInfo_passengerCount,ranker_id,requestDate,searchRoute,sex,taxes,totalPrice,selected
i64,bool,i64,str,i64,bool,str,str,str,str,str,str,f64,f64,f64,str,str,str,str,f64,str,str,str,f64,f64,f64,str,str,str,str,f64,str,str,str,f64,f64,f64,…,str,str,f64,f64,f64,str,str,str,str,f64,str,str,str,f64,f64,f64,str,str,str,str,f64,str,str,str,f64,f64,f64,f64,f64,i64,str,datetime[ns],str,bool,f64,f64,i64
143,true,53407,null,36,false,"""2024-05-23T14:50:00""","""2024-05-23T11:35:00""","""02:15:00""","""AT7""","""IKT""","""IKT""",0.0,0.0,1.0,"""KJA""","""02:15:00""","""UT""","""UT""",5.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,6000.0,1.0,0.0,0.0,1.0,1,"""e109b50aca4a43908dd146c55733e3…",2024-05-17 04:06:51,"""KJAIKT""",true,1215.0,3515.0,0
144,true,53407,null,36,false,"""2024-05-23T14:50:00""","""2024-05-23T11:35:00""","""02:15:00""","""AT7""","""IKT""","""IKT""",1.0,0.0,1.0,"""KJA""","""02:15:00""","""UT""","""UT""",5.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2000.0,1.0,0.0,0.0,1.0,1,"""e109b50aca4a43908dd146c55733e3…",2024-05-17 04:06:51,"""KJAIKT""",true,1215.0,5315.0,0
145,true,53407,null,36,false,"""2024-05-23T14:50:00""","""2024-05-23T11:35:00""","""02:15:00""","""AT7""","""IKT""","""IKT""",1.0,0.0,1.0,"""KJA""","""02:15:00""","""UT""","""UT""",5.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2000.0,1

In [5]:
uniqueRankerId = trainDFPl_base["ranker_id"].unique()

gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_idx, valid_idx = next(gss.split(uniqueRankerId, groups=uniqueRankerId))


train_rankers = uniqueRankerId[train_idx]
valid_rankers = uniqueRankerId[valid_idx]
print(len(train_rankers), len(valid_rankers))


validDFPl = trainDFPl_base.filter(pl.col("ranker_id").is_in(valid_rankers)).sort("ranker_id")
trainDFPl = trainDFPl_base.filter(pl.col("ranker_id").is_in(train_rankers)).sort("ranker_id")

23746 2639


/tmp/ipykernel_42534/2826069691.py:12: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  validDFPl = trainDFPl_base.filter(pl.col("ranker_id").is_in(valid_rankers)).sort("ranker_id")
/tmp/ipykernel_42534/2826069691.py:13: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  trainDFPl = trainDFPl_base.filter(pl.col("ranker_id").is_in(train_rankers)).sort("ranker_id")


In [6]:
trainDFPl_base, uniqueRankerId, gss , train_idx, valid_idx, train_rankers= None , None , None , None, None, None
del trainDFPl_base, uniqueRankerId, gss , train_idx, valid_idx, train_rankers
gc.collect()

16

# ===========================

In [7]:
class EnhancedRankingHeadFast(nn.Module):
    def __init__(self, input_dim, dropout=0.01, lstm_hidden=128, cnn_channels=64, kernel_size=3):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # 1D CNN to capture local feature interactions
        self.cnn = nn.Conv1d(
            in_channels=input_dim, 
            out_channels=cnn_channels, 
            kernel_size=kernel_size, 
            padding=1
        )

        # Bidirectional LSTM to capture sequence-level dependencies
        self.lstm = nn.LSTM(
            input_size=cnn_channels,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # Attention gating per flight
        #self.attention = nn.Sequential(
        #    nn.Linear(lstm_hidden*2, lstm_hidden*2),
        #    nn.Sigmoid()  # outputs values between 0 and 1 for gating
        #)
        self.attention = nn.Sequential(
            nn.Linear(lstm_hidden*2, 64),
            nn.Tanh(),
            nn.Linear(64, lstm_hidden*2)  # make sure output shape matches LSTM
        )

        # Fully connected layers for final ranking score
        self.fc = nn.Sequential(
            nn.Linear(lstm_hidden*2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x: (batch_size, seq_len, embedding_dim)
        x = x.transpose(1, 2)        # (batch_size, embedding_dim, seq_len) for CNN
        x = self.cnn(x)
        x = x.transpose(1, 2)        # (batch_size, seq_len, cnn_channels) for LSTM

        lstm_out, _ = self.lstm(x)   # (batch_size, seq_len, lstm_hidden*2)

        # Attention gating per flight
        gates = self.attention(lstm_out)  # (batch_size, seq_len, lstm_hidden*2)
        lstm_out = lstm_out * gates       # element-wise gate
        
        scores = self.fc(lstm_out).squeeze(-1)  # (batch_size, seq_len)
        return scores



class MiniLMFlightRanker(nn.Module):
    def __init__(self, dropout=0.1, freeze_lm=False, model_name="sentence-transformers/all-MiniLM-L6-v2"):
        super().__init__()
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.lm = SentenceTransformer(model_name)
        
        # Freeze LM if needed
        if freeze_lm:
            for p in self.lm.parameters():
                p.requires_grad = False

        hidden_size = self.lm.get_sentence_embedding_dimension()
        self.ranking_head = EnhancedRankingHeadFast(input_dim=hidden_size, dropout=dropout)

    def forward(self, flight_texts):
        embeddings = self.lm.encode(
            flight_texts, 
            convert_to_tensor=True, 
            device=self.device,
            show_progress_bar=False 
        )
        embeddings = embeddings.clone().detach().requires_grad_(True)
        embeddings = embeddings.unsqueeze(0) 
                
        scores = self.ranking_head(embeddings)
        #print(f"1. scores shape: {scores.shape}, {scores}")
        scores = scores.squeeze(0)  # (num_flights,)

        #print(f"2. len of flight_texts: {len(flight_texts)}")
        #print(f"3. embeddings shape: {embeddings.shape}")
        #print(f"4. scores shape after flatten: {scores.shape}, {scores}")
        #print()

        return scores  

# Create a encoder class which encodes (10, max_length) -> (10 row, each row encoded in max_legth) rows

In [8]:
import polars as pl
import torch

class FlightTextEncoder:
    def __init__(self, dfCols):
        self.dfCols = dfCols

    def encode_group(self, group_df: pl.DataFrame):
        # Replace NaNs and None with 'None', cast everything to string
        features_df = (
            group_df[self.dfCols]
            .fill_null("None")
            .fill_nan("None")
            .with_columns([pl.col(col).cast(pl.Utf8) for col in self.dfCols])
        )

        # Join columns into one string per row: "col1: val | col2: val | ..."
        flight_texts = (
            features_df
            .select(
                pl.concat_str(
                    [pl.lit(f"{col}: ") + pl.col(col) for col in self.dfCols],
                    separator=" | "
                )
                .alias("flight_text")
            )
            .to_series()
            .to_list()
        )

        # Safety check: no None values left
        flight_texts = ["" if txt is None else txt for txt in flight_texts]

        # Target: 1 in the position of selected row
        selectedCol = group_df["selected"].to_list()
        target = torch.zeros(len(selectedCol), dtype=torch.float32)
        target[selectedCol.index(1)] = 1

        return flight_texts, target


In [9]:
dflcols =['bySelf','companyID','frequentFlyer','nationality','isVip','legs0_arrivalAt','legs0_departureAt','legs0_duration',
 'legs0_segments0_aircraft_code','legs0_segments0_arrivalTo_airport_city_iata','legs0_segments0_arrivalTo_airport_iata','legs0_segments0_baggageAllowance_quantity',
 'legs0_segments0_baggageAllowance_weightMeasurementType','legs0_segments0_cabinClass','legs0_segments0_departureFrom_airport_iata',
 'legs0_segments0_duration','legs0_segments0_marketingCarrier_code','legs0_segments0_operatingCarrier_code','legs0_segments0_seatsAvailable',
 'legs0_segments1_aircraft_code','legs0_segments1_arrivalTo_airport_city_iata','legs0_segments1_arrivalTo_airport_iata','legs0_segments1_baggageAllowance_quantity',
 'legs0_segments1_baggageAllowance_weightMeasurementType','legs0_segments1_cabinClass','legs0_segments1_departureFrom_airport_iata',
 'legs0_segments1_duration','legs0_segments1_marketingCarrier_code','legs0_segments1_operatingCarrier_code','legs0_segments1_seatsAvailable',
 'legs0_segments2_aircraft_code','legs0_segments2_arrivalTo_airport_city_iata','legs0_segments2_arrivalTo_airport_iata','legs0_segments2_baggageAllowance_quantity',
 'legs0_segments2_baggageAllowance_weightMeasurementType','legs0_segments2_cabinClass','legs0_segments2_departureFrom_airport_iata',
 'legs0_segments2_duration','legs0_segments2_marketingCarrier_code','legs0_segments2_operatingCarrier_code','legs0_segments2_seatsAvailable',
 'legs0_segments3_aircraft_code','legs0_segments3_marketingCarrier_code','legs0_segments3_operatingCarrier_code','legs1_arrivalAt',
 'legs1_departureAt','legs1_duration','legs1_segments0_aircraft_code','legs1_segments0_arrivalTo_airport_city_iata','legs1_segments0_arrivalTo_airport_iata',
 'legs1_segments0_baggageAllowance_quantity','legs1_segments0_baggageAllowance_weightMeasurementType','legs1_segments0_cabinClass','legs1_segments0_departureFrom_airport_iata',
 'legs1_segments0_duration','legs1_segments0_marketingCarrier_code','legs1_segments0_operatingCarrier_code','legs1_segments0_seatsAvailable',
 'legs1_segments1_aircraft_code','legs1_segments1_arrivalTo_airport_city_iata','legs1_segments1_arrivalTo_airport_iata','legs1_segments1_baggageAllowance_quantity',
 'legs1_segments1_baggageAllowance_weightMeasurementType','legs1_segments1_cabinClass','legs1_segments1_departureFrom_airport_iata','legs1_segments1_duration',
 'legs1_segments1_marketingCarrier_code','legs1_segments1_operatingCarrier_code','legs1_segments1_seatsAvailable','legs1_segments2_aircraft_code',
 'legs1_segments2_arrivalTo_airport_city_iata','legs1_segments2_arrivalTo_airport_iata','legs1_segments2_baggageAllowance_quantity','legs1_segments2_baggageAllowance_weightMeasurementType',
 'legs1_segments2_cabinClass','legs1_segments2_departureFrom_airport_iata','legs1_segments2_duration','legs1_segments2_marketingCarrier_code',
 'legs1_segments2_operatingCarrier_code','legs1_segments2_seatsAvailable','legs1_segments3_aircraft_code','legs1_segments3_marketingCarrier_code',
 'legs1_segments3_operatingCarrier_code','miniRules0_monetaryAmount','miniRules0_statusInfos','miniRules1_monetaryAmount',
 'miniRules1_statusInfos','pricingInfo_isAccessTP','pricingInfo_passengerCount','requestDate','searchRoute','sex','taxes','totalPrice',
]

# 'Id','ranker_id','selected',

### Make a Evaluation fucniotn and pre-encode the filght info early to reduce the computation cost

In [10]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
flightEncoder = FlightTextEncoder(dflcols)

eval_ranker_embeddings = []


# eval_hit_k(model, flightEncoder, valid_rankers, validDFPl, device, k=3) 
for e_indx, evalRankerIds in enumerate(valid_rankers):
    eval_filtered_df = validDFPl.filter(
        pl.col("ranker_id") == evalRankerIds
    )
    flight_texts, eval_target = flightEncoder.encode_group(eval_filtered_df)
    eval_ranker_embeddings.append({
        "flight_texts": flight_texts,
        "eval_target": eval_target
    })


valid_rankers ,validDFPl = None, None 
del valid_rankers ,validDFPl
gc.collect()

0

In [11]:
def eval_hit_k(model, eval_ranker_embeddings, device, k=3):
    model.eval()
    total_hits = 0
    total_samples = 0

    with torch.no_grad():
        for e_indx, evalInfos in enumerate(eval_ranker_embeddings):
            flight_texts = evalInfos["flight_texts"]
            eval_target = evalInfos["eval_target"]

            scores = model(flight_texts)  # (num_flights,)
            
            # Ensure k is not larger than number of flights
            current_k = min(k, scores.size(0))
            top_k_indices = torch.topk(scores, current_k).indices
            selected_idx = torch.argmax(eval_target)
            
            # Hit@k check
            if selected_idx in top_k_indices:
                total_hits += 1
            total_samples += 1

            print(total_samples)

            # Cleanup
            del flight_texts, eval_target, scores, top_k_indices, selected_idx
            torch.cuda.empty_cache() if torch.cuda.is_available() else None
            gc.collect()

    hit_at_k = total_hits / total_samples
    return hit_at_k


## =========  Now Make Pairwise Loss Fn and start training ==========

In [12]:
def pairwise_ranking_loss(scores, labels):
    loss = torch.tensor(0.0, requires_grad=True)
    count = 0
    for i in range(len(scores)):
        for j in range(len(scores)):
            if labels[i] > labels[j]:
                loss = loss + torch.log1p(torch.exp(-(scores[i] - scores[j])))
                count += 1
    if count > 0:
        loss = loss / count
    return loss

In [ ]:
model = MiniLMFlightRanker(freeze_lm=False).to(device)
optimizer = torch.optim.AdamW(model.ranking_head.parameters(), lr=1e-4)

loss_fn = nn.CrossEntropyLoss()


for epoch in range(1):
    # run 10 epochs over the hole dataset
    model.train()
    total_loss_per_epoch = 0
    
    for dsLoop in range(1,5): # it is because I have separated my dataset in 4 parts and saved 3 part in kaggle output
        if dsLoop != 1: 
            trainDFPl = pl.read_parquet(f"./dataset_part{dsLoop}.parquet")
            print(f"Completed reading dataset part number:{dsLoop}")

        uniqueRanker = trainDFPl["ranker_id"].unique().to_list()
        
        for indx, rankerIds in enumerate(uniqueRanker): 
            model.train()
            filtered_df = trainDFPl.filter(
                pl.col("ranker_id") == rankerIds
            )
            
            flight_texts, target = flightEncoder.encode_group(filtered_df)
    
            optimizer.zero_grad()
            scores = model(flight_texts)
            loss = pairwise_ranking_loss(scores, target)
            # loss = loss_fn(scores.unsqueeze(0), target_idx)

            #print(f"scores shape: {scores.shape}")
            #print(f"target_idx shape: {target_idx.shape} , {target_idx}")
            #print(f"loss shape: {loss.shape}, {loss}")
                    
            loss.backward()
            optimizer.step()
    
            total_loss_per_epoch += loss.item()
            # print(f"Losssss: {loss.item():.4f}")
            
            if indx%10 == 0: 
                print(f"Epoch:{epoch} || ranker Index:{indx} || Loss:{loss.item():.4f}  || Avg Loss:{total_loss_per_epoch/(indx + 1):.4f}")

        
            del flight_texts, target, scores
            torch.cuda.empty_cache()
            gc.collect()

        evalHit_3 = eval_hit_k(model, eval_ranker_embeddings, device, k=3)
        print(f"Eval at Hitrate@3: {evalHit_3}")

        if dsLoop==2:  ## =============********* ================================
            break # at this moment I will train dataset 1 an 2 

        trainDFPl = None 
        del trainDFPl
        gc.collect()
        print(f"Completed dataset part number:{dsLoop}")
        

    
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    gc.collect()
    print(f"Loos in Epoch:{epoch} is -> {total_loss_per_epoch/len(uniqueRanker):.4f}")
        

Epoch:0 || ranker Index:0 || Loss:0.6921  || Avg Loss:0.6921


Epoch:0 || ranker Index:10 || Loss:0.6897  || Avg Loss:0.6923
Epoch:0 || ranker Index:20 || Loss:0.6924  || Avg Loss:0.6583
Epoch:0 || ranker Index:30 || Loss:0.6946  || Avg Loss:0.6694
Epoch:0 || ranker Index:40 || Loss:0.6984  || Avg Loss:0.6750
Epoch:0 || ranker Index:50 || Loss:0.6965  || Avg Loss:0.6789
Epoch:0 || ranker Index:60 || Loss:0.6991  || Avg Loss:0.6813
Epoch:0 || ranker Index:70 || Loss:0.6911  || Avg Loss:0.6827
Epoch:0 || ranker Index:80 || Loss:0.6957  || Avg Loss:0.6840
Epoch:0 || ranker Index:90 || Loss:0.6979  || Avg Loss:0.6853
Epoch:0 || ranker Index:100 || Loss:0.6928  || Avg Loss:0.6861
Epoch:0 || ranker Index:110 || Loss:0.6911  || Avg Loss:0.6864
Epoch:0 || ranker Index:120 || Loss:0.6893  || Avg Loss:0.6869
Epoch:0 || ranker Index:130 || Loss:0.6985  || Avg Loss:0.6874
Epoch:0 || ranker Index:140 || Loss:0.6917  || Avg Loss:0.6876
Epoch:0 || ranker Index:150 || Loss:0.6941  || Avg Loss:0.6835
Epoch:0 || ranker Index:160 || Loss:0.6974  || Avg Loss:0.6836
E

# Now save the model

In [ ]:
# Save the model state and ptimizer state if you want to resume exactly from this training point
torch.save(model.state_dict(), "minilm_ranker.pt")
torch.save(optimizer.state_dict(), "optimizer.pt")

# Later load the model and optimizer to start the model train/testagain

In [ ]:
# Recreate the model architecture
model = MiniLMFlightRanker(
    dropout=0.1, 
    freeze_lm=False,  # Set same as before
    model_name="sentence-transformers/all-MiniLM-L6-v2"
).to(device)

# Load saved weights
model.load_state_dict(torch.load("minilm_ranker.pt", map_location=device))

# Recreate optimizer and load its state
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
optimizer.load_state_dict(torch.load("optimizer.pt", map_location=device))

# Now you can keep training
model.train()